# Baseline Models

model comparison with:
- a naive persistence baseline
- linear regression
- random forest
- gradient boosting

Goal: compare simple and classical machine-learning baselines for
60-minute-ahead household power forecasting.

Evaluation:
- MAE
- RMSE
- R²

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

In [2]:
data_path = Path(
    "../data/processed/household_power_features.parquet"
)

df = pd.read_parquet(data_path)

df = df.sort_values("datetime").reset_index(drop=True)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(1365170, 24)
(292537, 24)
(292537, 24)


In [3]:
feature_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    "power_lag_1",
    "power_lag_5",
    "power_lag_15",
    "power_lag_30",
    "power_lag_60",
    "power_lag_120",
    "power_lag_1380",

    "hour",
    "day_of_week",
    "month",
    "is_weekend",

    "rolling_mean_15",
    "rolling_mean_60",
    "rolling_std_60",
    "rolling_mean_1440"
]

target_col = "power_target_60"

In [4]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

In [5]:
def evaluate_regression(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(
        y_true,
        y_pred
    ) ** 0.5
    r2 = r2_score(y_true, y_pred)

    return {
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [6]:
results = []